# stockcorr — Exploration Notebook

Walks through every metric category on a small universe so the output of each step is easy to inspect.

Universe: by default we use a 12-ticker subset (mix of S&P 500 + NDX + ADR). Swap `TICKERS = union_universe()['ticker'].tolist()` to scale to the full ~614-ticker universe.

Note: `yfinance` requires outbound network. If this runs in a sandbox without Yahoo Finance access, set `USE_SYNTHETIC = True` to inject a synthetic panel for testing the pipeline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from stockcorr.data import YFinanceSource
from stockcorr.data.universe import union_universe, sp500_tickers, ndx_tickers, top_adrs
from stockcorr.pipeline import run_metrics
from stockcorr.viz import plot_heatmap, plot_network, plot_spread, plot_rolling_corr
from stockcorr.metrics.base import pairs

USE_SYNTHETIC = False
TICKERS = ['AAPL','MSFT','GOOGL','META','NVDA','AMD','TSM','ASML','KO','PEP','JPM','BAC','XOM','CVX']
START, END = '2020-01-01', '2024-12-31'
BENCH = 'SPY'
print('universe size:', len(TICKERS))

## 1. Load prices

In [ ]:
if USE_SYNTHETIC:
    rng = np.random.default_rng(0)
    n = 1000
    dates = pd.bdate_range(START, periods=n)
    factors = rng.standard_normal((n, 3)) * 0.01
    panel_data = {}
    for t in TICKERS:
        load = rng.standard_normal(3)
        eps = rng.normal(0, 0.005, n)
        r = factors @ load + eps
        panel_data[t] = 100 * np.exp(np.cumsum(r))
    panel = pd.DataFrame(panel_data, index=dates)
    bench = pd.Series(100 * np.exp(np.cumsum(factors[:, 0])), index=dates, name=BENCH)
else:
    src = YFinanceSource()
    panel = src.close_panel(TICKERS, START, END)
    bench_panel = src.close_panel([BENCH], START, END)
    bench = bench_panel.iloc[:, 0].rename(BENCH) if not bench_panel.empty else None
print('panel shape:', panel.shape)
panel.head()

## 2. Linear baseline — Pearson

In [ ]:
results = run_metrics(panel, ['pearson', 'spearman'])
plot_heatmap(results, 'pearson')
results[results['metric'] == 'pearson'].sort_values('value', ascending=False).head(10)

## 3. Cointegration scan with Pearson pre-filter

In [ ]:
co = run_metrics(
    panel,
    ['coint', 'half_life', 'hurst'],
    prefilter={'metric': 'pearson', 'min_abs_value': 0.5},
    n_jobs=-1,
)
co[co['metric'] == 'coint'].sort_values('p_value').head(10)

In [ ]:
# Inspect the most cointegrated pair: spread + rolling corr
top_pair = co[co['metric'] == 'coint'].sort_values('p_value').iloc[0]
a, b = top_pair['ticker_a'], top_pair['ticker_b']
print(f'top cointegrated pair: {a} - {b} (p={top_pair["p_value"]:.4f})')
plot_spread(panel, a, b)
plot_rolling_corr(panel, a, b, window=60)

## 4. Lead-lag (cross-correlation) and Granger causality

In [ ]:
ll = run_metrics(panel, ['cross_corr'], prefilter={'metric': 'pearson', 'min_abs_value': 0.3}, n_jobs=-1)
cc = ll[ll['metric'] == 'cross_corr']
# Pairs where best correlation is at non-zero lag indicate lead-lag
cc.sort_values('lag', key=lambda s: s.abs(), ascending=False).head(10)

## 5. Factor cleanup — residual correlation

In [ ]:
if bench is not None:
    fc = run_metrics(panel, ['beta', 'residual_corr'], benchmark=bench)
    print(fc[fc['metric'] == 'beta'].sort_values('value', ascending=False).head(10))
    plot_heatmap(fc, 'residual_corr')

## 6. Volatility / tail dependence

In [ ]:
vt = run_metrics(panel, ['vol_corr', 'tail_dep'], prefilter={'metric': 'pearson', 'min_abs_value': 0.3}, n_jobs=-1)
vt[vt['metric'] == 'tail_dep'].sort_values('value', ascending=False).head(10)

## 7. Nonlinear dependence

In [ ]:
nl = run_metrics(panel, ['distance_corr', 'mutual_info'], prefilter={'metric': 'pearson', 'min_abs_value': 0.3}, n_jobs=-1)
# Scatter dcor vs pearson
p = results[results['metric'] == 'pearson'].set_index(['ticker_a', 'ticker_b'])['value']
d = nl[nl['metric'] == 'distance_corr'].set_index(['ticker_a', 'ticker_b'])['value']
common = p.index.intersection(d.index)
plt.figure(figsize=(6, 5))
plt.scatter(p.loc[common].abs(), d.loc[common], alpha=0.6)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.3)
plt.xlabel('|Pearson|'); plt.ylabel('Distance correlation')
plt.title('Nonlinear vs linear dependence')
plt.show()

## 8. Hierarchical clustering

In [ ]:
cl = run_metrics(panel, ['cluster'], metric_kwargs={'cluster': {'n_clusters': 4}})
cl.sort_values('value')

## 9. Top-K network of strong relationships

In [ ]:
plot_network(results, 'pearson', top=40)

## 10. Export to parquet for CLI plotting

In [ ]:
all_results = pd.concat([results, co, ll, vt, nl, cl], ignore_index=True, sort=False)
all_results.to_parquet('exploration_results.parquet')
print('rows:', len(all_results))
all_results.groupby('metric').size()